# European Employment by Sex — Data Cleaning

This notebook cleans and prepares the Eurostat employment dataset for exploratory analysis and visualisation.

Cleaning steps:

- preserve the original dataset
- remove empty and constant metadata columns
- retain and document observation flags
- standardise column names
- validate the cleaned dataset
- save the cleaned data

In [20]:
import pandas as pd 
df_raw = pd.read_csv("data/employment_by_sex.csv")
df_raw.head()

,DATAFLOW,LAST UPDATE,freq,indic_em,unit,age,sex,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:TESEM010(1.0),29/04/26 23:00:00,Annual,Total employment (resident population concept ...,Percentage of total population,From 20 to 64 years,Females,Austria,2009,68.2,NaN,NaN
1,ESTAT:TESEM010(1.0),29/04/26 23:00:00,Annual,Total employment (resident population concept ...,Percentage of total population,From 20 to 64 years,Females,Austria,2010,68.8,NaN,NaN
2,ESTAT:TESEM010(1.0),29/04/26 23:00:00,Annual,Total employment (resident population concept ...,Percentage of total population,From 20 to 64 years,Females,Austria,2011,69.2,NaN,NaN
3,ESTAT:TESEM010(1.0),29/04/26 23:00:00,Annual,Total employment (resident population concept ...,Percentage of total population,From 20 to 64 years,Females,Austria,2012,69.6,NaN,NaN
4,ESTAT:TESEM010(1.0),29/04/26 23:00:00,Annual,Total employment (resident population concept ...,Percentage of total population,From 20 to 64 years,Females,Austria,2013,70.0,NaN,NaN


In [21]:
df_clean = df_raw.copy()
df_clean.shape

(1896, 12)

In [22]:
df_clean = df_clean.drop(columns=["CONF_STATUS"])
df_clean.shape

(1896, 11)

In [23]:
constant_columns = [
    "DATAFLOW",
    "LAST UPDATE", 
    "freq",
    "indic_em",
    "unit",
    "age"
]
df_clean = df_clean.drop(columns=constant_columns)
df_clean.head()

,sex,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG
0,Females,Austria,2009,68.2,NaN
1,Females,Austria,2010,68.8,NaN
2,Females,Austria,2011,69.2,NaN
3,Females,Austria,2012,69.6,NaN
4,Females,Austria,2013,70.0,NaN


In [25]:
df_clean = df_clean.rename(
    columns={
        "geo":"country",
        "TIME_PERIOD":"year",
        "OBS_VALUE":"employment_rate",
        "OBS_FLAG":"observation_flag"
    }
)
print("columns:", df_clean.columns.tolist())

columns: ['sex', 'country', 'year', 'employment_rate', 'observation_flag']


In [26]:
df_clean["observation_flag"] = (
    df_clean["observation_flag"].fillna("none")
)
df_clean["observation_flag"].value_counts()

observation_flag
none    1794
b         72
d         30
Name: count, dtype: int64

In [27]:
df_clean.shape

(1896, 5)

In [29]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1896 entries, 0 to 1895
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sex               1896 non-null   str    
 1   country           1896 non-null   str    
 2   year              1896 non-null   int64  
 3   employment_rate   1896 non-null   float64
 4   observation_flag  1896 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 74.2 KB


In [30]:
df_clean.isna().sum()

sex                 0
country             0
year                0
employment_rate     0
observation_flag    0
dtype: int64

In [31]:
df_clean.duplicated().sum()

np.int64(0)

In [32]:
df_clean.duplicated(
    subset=["country", "year", "sex"]
).sum()

np.int64(0)

In [33]:
df_clean["employment_rate"].between(0,100).all()

np.True_

In [34]:
df_clean.to_csv(
    "data/processed/employment_by_sex_clean.csv",
    index=False         
)

In [35]:
saved_df = pd.read_csv("data/processed/employment_by_sex_clean.csv")
saved_df.head()

,sex,country,year,employment_rate,observation_flag
0,Females,Austria,2009,68.2,none
1,Females,Austria,2010,68.8,none
2,Females,Austria,2011,69.2,none
3,Females,Austria,2012,69.6,none
4,Females,Austria,2013,70.0,none


In [36]:
saved_df.shape

(1896, 5)

## Cleaning summary

- The original dataset was preserved as `df_raw`.
- One completely empty column was removed.
- Six constant metadata columns were removed.
- Analytical columns were renamed using consistent snake_case naming.
- Missing observation flags were labelled as `none`.
- No duplicate country-year-sex records were found.
- Employment rates were confirmed to be within the valid 0–100 range.
- The cleaned dataset contains 1,896 rows and 5 columns.